# dolcestat — Quickstart 🍬

*Sweet statistics* — machine-learning algorithms built from scratch on NumPy.

This notebook is the **60-second tour**. Every model in `dolcestat` follows the
same three-step rhythm:

1. **Preprocess** — wrap your data in a `DolceSet`.
2. **Fit** — hand that `DolceSet` to a model.
3. **Evaluate** — ask the model for an *analyzer* and read off the metrics.

Once you've seen the shape of it here, the numbered notebooks go deep on each
piece. Let's fit a linear regression end to end.

## 1 · Preprocess

We'll predict apartment **price** (thousands of €) from its **size** and number
of **rooms**. Any tabular data works — here we generate a synthetic sample so
the notebook is self-contained.

A `DolceSet` is the container every model consumes. You load a
[Polars](https://pola.rs) DataFrame (or a plain dict) and name the target
column; it splits the features `X` from the target `y` for you.

In [1]:
import numpy as np
import polars as pl

from dolcestat.preprocessing import DolceSet

rng = np.random.default_rng(0)
n = 200
size_m2 = rng.uniform(40, 200, n)
rooms = rng.integers(1, 6, n).astype(float)
price_k = 3.0 * size_m2 + 25 * rooms + 50 + rng.normal(0, 15, n)

df = pl.DataFrame({"size_m2": size_m2, "rooms": rooms, "price_k": price_k})

data = DolceSet()
data.load_from_polars_dataframe(df, target_col="price_k")
data.get_features(), data.X.shape, data.y.shape

(['size_m2', 'rooms'], (200, 2), (200,))

## 2 · Fit

`LinearRegression` defaults to the **closed-form** solution (the normal
equation) — no hyperparameters to tune. Call `.fit(data)` and you're done.

In [7]:
from dolcestat.linear_models import LinearRegression

model = LinearRegression()
model.fit(data)
print(f"weights: {[x[0] for x in model.weights]}, bias: {model.bias}")  # [coef_size, coef_rooms, intercept]

weights: [2.987230122726995, 23.900722353407808], bias: [54.92803009]


## 3 · Evaluate

`model.predict(data)` returns an **analyzer**: an object holding the predictions
that computes metrics on demand. For a regression that's R², RMSE, and friends.
(You'll meet the full metrics toolkit in
[`06_metrics`](06_metrics.ipynb).)

In [8]:
report = model.predict(data)
print(f"R²   = {report.r2():.3f}")
print(f"RMSE = {report.rmse():.2f} (thousand €)")

R²   = 0.990
RMSE = 14.89 (thousand €)


## Where to go next

That's the whole pipeline. Each notebook below zooms in on one piece:

| Notebook | You'll learn |
|---|---|
| [`01_preprocessing`](01_preprocessing.ipynb) | Loading, encoding categoricals, and scaling with `DolceSet`. |
| [`02_linear_regression`](02_linear_regression.ipynb) | Linear regression: closed-form vs. gradient descent. |
| [`03_logistic_regression`](03_logistic_regression.ipynb) | Classification, the sigmoid, and Newton's method. |
| [`04_optimization`](04_optimization.ipynb) | The optimizers under the hood, on their own. |
| [`05_knn`](05_knn.ipynb) | k-nearest-neighbours for classification and regression. |
| [`06_metrics`](06_metrics.ipynb) | Evaluating models and choosing the right metric. |